In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from tabulate import tabulate
from tqdm import tqdm
from pycoingecko import CoinGeckoAPI
from datetime import datetime
import ta
import plotly.express as px

#####  A fazer: 

1. Trend stability
2. Support / resistance levels 
3. Implement ability to short

In [ ]:

class TrendAnalyzer:
    """
    A class to analyze price trends for multiple cryptocurrency assets, storing all data and classifications.

    Attributes:
        asset_ids (list): List of asset IDs to analyze
        data_path (str): Path to the directory containing candle data
        btc_data_path (str): Path to the Bitcoin data file
        use_btc_adjusted (bool): Whether to prioritize BTC-adjusted prices in output (default: True)
        verbose (bool): Whether to print detailed processing information (default: True)
        ticker_mapping (dict): Mapping from ticker symbols to asset IDs
        asset_data (dict): Dictionary storing raw data, indicators, classifications, and signals for each asset
        current_date (datetime): Current date for data currency validation
    """

    def __init__(self, asset_ids, data_path, btc_data_path, use_btc_adjusted=True, verbose=True):
        """
        Initialize the TrendAnalyzer with a list of asset IDs.

        Args:
            asset_ids (list): List of asset IDs to analyze
            data_path (str): Path to the directory containing candle data
            btc_data_path (str): Path to the Bitcoin data file
            use_btc_adjusted (bool): Whether to prioritize BTC-adjusted prices in output (default: True)
            verbose (bool): Whether to print detailed processing information (default: True)
        """
        self.asset_ids = asset_ids
        self.data_path = data_path
        self.btc_data_path = btc_data_path
        self.use_btc_adjusted = use_btc_adjusted
        self.verbose = verbose
        self.ma_periods = {
            'Short Term': [3, 5, 7, 14],
            'Medium Term': [21, 30, 45, 63],
            'Long Term': [84, 100, 120, 150, 200, 252, 365]
        }
        self.ticker_mapping = self._get_ticker_mapping()
        self.asset_data = {}  # Initialize state dictionary
        self.current_date = datetime(2025, 3, 11)  # Set current date as specified

    def _get_ticker_mapping(self):
        """
        Retrieve ticker mapping using CoinGecko API.

        Returns:
            dict: Mapping from ticker symbols (upper-case) to asset IDs
        """
        cg = CoinGeckoAPI()
        coins_list = cg.get_coins_list()
        coins_df = pd.DataFrame(coins_list)
        
        known_mappings = {
            'VIRTUAL': 'virtual-protocol',
            'HYPE': 'hyperliquid',
            'YNE': 'yesnoerror'
        }
        
        mapping = {}
        for ticker, coin_id in known_mappings.items():
            if coin_id in self.asset_ids:
                mapping[ticker] = coin_id
        
        filtered_coins_df = coins_df[coins_df['id'].isin(self.asset_ids)]
        for _, row in filtered_coins_df.iterrows():
            ticker = row['symbol'].upper()
            if ticker not in mapping:
                mapping[ticker] = row['id']
        
        return mapping

    def _load_data(self, asset_id):
        """
        Load and merge asset data with Bitcoin data if applicable.

        Args:
            asset_id (str): Asset ID to load data for

        Returns:
            pd.DataFrame: Loaded and processed raw data, or empty DataFrame if file not found
        """
        data_path = f"{self.data_path}{asset_id}_candles.csv"
        try:
            data = pd.read_csv(data_path)
        except FileNotFoundError:
            if self.verbose:
                print(f"Warning: File not found for {asset_id}: {data_path}")
            return pd.DataFrame()
        
        data['date'] = pd.to_datetime(data['date'])
        data.dropna(inplace=True)
        #if self.verbose:
        #    print(f"Initial columns after loading asset data for {asset_id}: {list(data.columns)}")

        if asset_id != 'bitcoin':
            btc_data = pd.read_csv(self.btc_data_path)
            btc_data['date'] = pd.to_datetime(btc_data['date'])
            btc_data = btc_data[['date', 'close']].rename(columns={'close': 'btc_close'})
            #if self.verbose:
            #    print(f"Bitcoin data columns: {list(btc_data.columns)}")
            data = data.merge(btc_data, on='date', how='left')
            #if self.verbose:
            #    print(f"Columns after merge for {asset_id}: {list(data.columns)}")
            if 'close' in data.columns and 'btc_close' in data.columns:
                data[f'{asset_id}_btc'] = data['close'] / data['btc_close']
            else:
                if self.verbose:
                    print(f"Missing columns after merge for {asset_id}. Available columns: {list(data.columns)}")
            data = data.drop(columns=['btc_close'], errors='ignore')
            #if self.verbose:
             #   print(f"Columns after dropping 'btc_close' for {asset_id}: {list(data.columns)}")

        # Check if data is current
        if not data.empty:
            latest_date = data['date'].max()
            #if self.verbose and (self.current_date - latest_date).days > 7:
            #    print(f"Warning: Data for {asset_id} is outdated (latest date: {latest_date.date()}, current date: {self.current_date.date()})")
        
        return data

    def _calculate_indicators(self, data, price_column, asset_id, suffix=''):
        """
        Calculate Exponential Moving Averages (EMAs) and Rate of Change (RoC) indicators for a given price column.

        Args:
            data (pd.DataFrame): DataFrame to calculate indicators for
            price_column (str): Name of the price column to use
            asset_id (str): Asset ID being processed
            suffix (str): Suffix to append to indicator column names (e.g., 'USD', 'BTC')
        """
        if price_column not in data.columns and not data.empty:
            raise KeyError(f"Column '{price_column}' not found for {asset_id}. Available columns: {list(data.columns)}")
        if not data.empty:
            data[price_column] = pd.to_numeric(data[price_column], errors='coerce')
            data.dropna(subset=[price_column], inplace=True)
           # if self.verbose:
           #     print(f"Columns after numeric conversion and dropna for {asset_id} ({price_column}): {list(data.columns)}")
            
            for term, periods in self.ma_periods.items():
                for period in periods:
                    data[f'EMA_{suffix}_{period}'] = data[price_column].ewm(
                        span=period, min_periods=period, adjust=False
                    ).mean()
            for term, periods in self.ma_periods.items():
                for period in periods:
                    data[f'RoC_{suffix}_{period}'] = (
                        (data[price_column] - data[price_column].shift(period))
                        / data[price_column].shift(period) * 100
                    )
          #  if self.verbose:
          #      print(f"Columns after calculating indicators for {asset_id} ({suffix}): {list(data.columns)}")

    def classify_trend(self, mas, rocs, data, suffix):
        """
        Classify the trend based on moving averages and rate of change for a specific price column.

        Args:
            mas (pd.Series): Series of moving averages
            rocs (pd.Series): Series of rates of change
            data (pd.DataFrame): DataFrame containing the asset's price data
            suffix (str): Suffix indicating the price basis (e.g., 'USD', 'BTC')

        Returns:
            str: Trend classification ('Strong Bull', 'Weak Bull', 'Strong Bear', 'Weak Bear', 'Neutral')
        """
        n_mas = len(mas.dropna())
        n_rocs = len(rocs.dropna())
        if n_mas == 0 or n_rocs == 0:
            return "Neutral"
        
        pos_mas = 0
        for col in mas.index:
            period = int(col.replace(f'SMA_{suffix}_', ''))
            current_ma = data[f'SMA_{suffix}_{period}'].iloc[-1]
            prev_ma = data[f'SMA_{suffix}_{period}'].iloc[-2] if len(data) > 1 else np.nan
            if not np.isnan(current_ma) and not np.isnan(prev_ma) and current_ma > prev_ma:
                pos_mas += 1
        
        pos_mas = pos_mas / n_mas * 100 if n_mas > 0 else 0
        pos_rocs = sum(1 for roc in rocs if roc > 0) / n_rocs * 100 if n_rocs > 0 else 0
        
        avg_pos = (pos_mas + pos_rocs) / 2
        
        if avg_pos >= 75:
            return "Strong Bull"
        elif avg_pos >= 50:
            return "Weak Bull"
        elif avg_pos <= 25:
            return "Strong Bear"
        elif avg_pos <= 40:
            return "Weak Bear"
        return "Neutral"

    def _create_classified_data(self, data, asset_id):
        """
        Create a DataFrame with historical trend classifications for both USD and BTC price columns.

        Args:
            data (pd.DataFrame): DataFrame with price and indicator data
            asset_id (str): Asset ID being processed

        Returns:
            pd.DataFrame: DataFrame with trend classifications for both USD and BTC
        """
        if data.empty:
            return pd.DataFrame()
        classified_data = data.copy()

        # USD classifications
        usd_price_column = 'close'
        self._calculate_indicators(classified_data, usd_price_column, asset_id, suffix='USD')
        
        short_term_mas_usd = [f'SMA_USD_{period}' for period in self.ma_periods['Short Term']]
        medium_term_mas_usd = [f'SMA_USD_{period}' for period in self.ma_periods['Medium Term']]
        long_term_mas_usd = [f'SMA_USD_{period}' for period in self.ma_periods['Long Term']]
        
        short_term_rocs_usd = [f'RoC_USD_{period}' for period in self.ma_periods['Short Term']]
        medium_term_rocs_usd = [f'RoC_USD_{period}' for period in self.ma_periods['Medium Term']]
        long_term_rocs_usd = [f'RoC_USD_{period}' for period in self.ma_periods['Long Term']]

        classified_data['Short Term (USD)'] = classified_data.apply(
            lambda row: self.classify_trend(row[short_term_mas_usd], row[short_term_rocs_usd], classified_data, 'USD'), axis=1
        )
        classified_data['Medium Term (USD)'] = classified_data.apply(
            lambda row: self.classify_trend(row[medium_term_mas_usd], row[medium_term_rocs_usd], classified_data, 'USD'), axis=1
        )
        classified_data['Long Term (USD)'] = classified_data.apply(
            lambda row: self.classify_trend(row[long_term_mas_usd], row[long_term_rocs_usd], classified_data, 'USD'), axis=1
        )
        classified_data['Overall (USD)'] = classified_data.apply(
            lambda row: self._compute_overall_classification(
                [row['Short Term (USD)'], row['Medium Term (USD)'], row['Long Term (USD)']]
            ), axis=1
        )

        # BTC classifications (except for Bitcoin)
        if asset_id != 'bitcoin' and f'{asset_id}_btc' in classified_data.columns:
            btc_price_column = f'{asset_id}_btc'
            self._calculate_indicators(classified_data, btc_price_column, asset_id, suffix='BTC')
            
            short_term_mas_btc = [f'SMA_BTC_{period}' for period in self.ma_periods['Short Term']]
            medium_term_mas_btc = [f'SMA_BTC_{period}' for period in self.ma_periods['Medium Term']]
            long_term_mas_btc = [f'SMA_BTC_{period}' for period in self.ma_periods['Long Term']]
            
            short_term_rocs_btc = [f'RoC_BTC_{period}' for period in self.ma_periods['Short Term']]
            medium_term_rocs_btc = [f'RoC_BTC_{period}' for period in self.ma_periods['Medium Term']]
            long_term_rocs_btc = [f'RoC_BTC_{period}' for period in self.ma_periods['Long Term']]

            classified_data['Short Term (BTC)'] = classified_data.apply(
                lambda row: self.classify_trend(row[short_term_mas_btc], row[short_term_rocs_btc], classified_data, 'BTC'), axis=1
            )
            classified_data['Medium Term (BTC)'] = classified_data.apply(
                lambda row: self.classify_trend(row[medium_term_mas_btc], row[medium_term_rocs_btc], classified_data, 'BTC'), axis=1
            )
            classified_data['Long Term (BTC)'] = classified_data.apply(
                lambda row: self.classify_trend(row[long_term_mas_btc], row[long_term_rocs_btc], classified_data, 'BTC'), axis=1
            )
            classified_data['Overall (BTC)'] = classified_data.apply(
                lambda row: self._compute_overall_classification(
                    [row['Short Term (BTC)'], row['Medium Term (BTC)'], row['Long Term (BTC)']]
                ), axis=1
            )
        else:
            # For Bitcoin, set BTC trends to NaN
            classified_data['Short Term (BTC)'] = pd.NA
            classified_data['Medium Term (BTC)'] = pd.NA
            classified_data['Long Term (BTC)'] = pd.NA
            classified_data['Overall (BTC)'] = pd.NA
        
        return classified_data

    def _compute_overall_classification(self, trends):
        """
        Compute the overall trend classification based on individual timeframes.

        Args:
            trends (list): List of trend classifications for Short Term, Medium Term, and Long Term

        Returns:
            str: Overall trend classification
        """
        overall_pos = sum(1 for c in trends if c in ["Strong Bull", "Weak Bull"]) / len(trends) * 100
        if overall_pos >= 75:
            return "Strong Bull"
        elif overall_pos >= 50:
            return "Weak Bull"
        elif overall_pos <= 25:
            return "Strong Bear"
        elif overall_pos <= 40:
            return "Weak Bear"
        return "Neutral"

    def analyze(self, asset_id, data, suffix):
        """
        Analyze trends for a single asset using a specific price suffix.

        Args:
            asset_id (str): Asset ID to analyze
            data (pd.DataFrame): DataFrame containing price data
            suffix (str): Suffix indicating the price basis (e.g., 'USD', 'BTC')

        Returns:
            dict: Dictionary of trend classifications for Short Term, Medium Term, Long Term, and Overall
        """
        if data.empty:
            return {'Short Term': 'N/A', 'Medium Term': 'N/A', 'Long Term': 'N/A', 'Overall': 'N/A'}
        classifications = {}
        for term in ['Short Term', 'Medium Term', 'Long Term']:
            classifications[term] = data[f'{term} ({suffix})'].iloc[-1]
        classifications['Overall'] = data[f'Overall ({suffix})'].iloc[-1]
        if self.verbose:
            for term, classification in classifications.items():
                print(f"{term} classification for {asset_id} ({suffix}): {classification}")
        return classifications

    def create_chart(self, asset_id, data, price_column):
        """
        Create a Plotly chart of price and SMAs for a single asset.

        Args:
            asset_id (str): Asset ID to create chart for
            data (pd.DataFrame): DataFrame containing price and SMA data
            price_column (str): Name of the price column to plot

        Returns:
            go.Figure: Plotly figure object, or None if no data
        """
        if data.empty:
            if self.verbose:
                print(f"No data available to create chart for {asset_id} ({price_column})")
            return None
        selected_smas = [14, 30, 63, 200]
        suffix = 'USD' if price_column == 'close' else 'BTC'
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=data['date'],
            y=data[price_column],
            mode='lines',
            name='Price',
            line=dict(color='green'),
            yaxis='y1'
        ))

        colors = ['red', 'orange', 'purple', 'gray']
        for i, period in enumerate(selected_smas):
            fig.add_trace(go.Scatter(
                x=data['date'],
                y=data[f'SMA_{suffix}_{period}'],
                mode='lines',
                name=f'SMA{period}',
                line=dict(color=colors[i % len(colors)], width=2),
                yaxis='y1'
            ))

        fig.update_layout(
            title=f"{asset_id.upper()} Price and SMAs ({suffix})",
            xaxis_title="Date",
            yaxis_title=f"Price ({suffix})",
            yaxis=dict(type='log', autorange=True, gridcolor='lightgray'),
            xaxis=dict(gridcolor='lightgray'),
            template="plotly_white",
            showlegend=True,
            height=500,
            margin=dict(l=50, r=50, t=100, b=50)
        )
        return fig

    def analyze_multiple_assets(self):
        """
        Analyze trends for all assets, store data in state, and output a consolidated table using ticker symbols.

        Returns:
            pd.DataFrame: DataFrame containing the most recent trend analysis for all assets
        """
        self.asset_data = {}  # Initialize state dictionary
        results = []
        reverse_mapping = {v: k for k, v in self.ticker_mapping.items()}
        
        for asset_id in tqdm(self.asset_ids, desc="Analyzing assets", disable=not self.verbose):
            # Load raw data
            raw_data = self._load_data(asset_id)
            
            # Process classifications for both USD and BTC in a single DataFrame
            classified_data = self._create_classified_data(raw_data, asset_id)

            # Store all data in state
            self.asset_data[asset_id] = {
                'raw_data': raw_data.copy(),
                'classified_data': classified_data.copy(),
                'price_column_usd': 'close',
                'price_column_btc': f'{asset_id}_btc' if asset_id != 'bitcoin' and f'{asset_id}_btc' in classified_data.columns else None
            }

            # Get the most recent classifications
            latest_classifications = classified_data.iloc[-1].to_dict() if not classified_data.empty else {
                'Short Term (USD)': 'N/A',
                'Medium Term (USD)': 'N/A',
                'Long Term (USD)': 'N/A',
                'Overall (USD)': 'N/A',
                'Short Term (BTC)': 'N/A',
                'Medium Term (BTC)': 'N/A',
                'Long Term (BTC)': 'N/A',
                'Overall (BTC)': 'N/A'
            }

            ticker = reverse_mapping.get(asset_id, asset_id.upper())
            results.append({
                'Ticker': ticker,
                'Short Term Trend (USD)': latest_classifications.get('Short Term (USD)', 'N/A'),
                'Medium Term Trend (USD)': latest_classifications.get('Medium Term (USD)', 'N/A'),
                'Long Term Trend (USD)': latest_classifications.get('Long Term (USD)', 'N/A'),
                'Overall Trend (USD)': latest_classifications.get('Overall (USD)', 'N/A'),
                'Short Term Trend (BTC)': latest_classifications.get('Short Term (BTC)', 'N/A'),
                'Medium Term Trend (BTC)': latest_classifications.get('Medium Term (BTC)', 'N/A'),
                'Long Term Trend (BTC)': latest_classifications.get('Long Term (BTC)', 'N/A'),
                'Overall Trend (BTC)': latest_classifications.get('Overall (BTC)', 'N/A'),
                'Latest Date': classified_data['date'].iloc[-1].date() if not classified_data.empty else None
            })

        summary_df = pd.DataFrame(results)
        
        if self.verbose:
            print("\nMost Recent Trend Analysis Summary for Multiple Assets (as of March 11, 2025):")
            print(tabulate(summary_df, headers='keys', tablefmt='pretty', showindex=False))
        
        return summary_df

# Usage
if __name__ == "__main__":
    from scripts.assetsRoster import carteira_AC, carteira_HB, carteira_LC, carteira_EXC, others

    # Combine all unique assets from the portfolio
    all_assets = list(set(carteira_AC + carteira_HB + carteira_LC + carteira_EXC + others))
    
    analyzer = TrendAnalyzer(
        asset_ids=all_assets,
        data_path='/Users/valter.rebelo/MissionControl/data/micro/candleData/',
        btc_data_path='/Users/valter.rebelo/MissionControl/data/micro/assetData/bitcoin.csv',
        use_btc_adjusted=False,
        verbose=True
    )
    summary_df = analyzer.analyze_multiple_assets()

 

In [433]:
summary_df

,Ticker,Short Term Trend (USD),Medium Term Trend (USD),Long Term Trend (USD),Overall Trend (USD),Short Term Trend (BTC),Medium Term Trend (BTC),Long Term Trend (BTC),Overall Trend (BTC),Latest Date
0,ML,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,2025-03-18
1,CRO,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Strong Bull,Weak Bull,Strong Bear,Weak Bull,2025-03-18
2,ETHFI,Weak Bull,Strong Bear,Strong Bear,Weak Bear,Weak Bull,Strong Bear,Strong Bear,Weak Bear,2025-03-18
3,POLY,Weak Bull,Weak Bear,Strong Bear,Weak Bear,Strong Bull,Weak Bull,Strong Bear,Weak Bull,2025-03-18
4,AAVE,Weak Bull,Strong Bear,Weak Bear,Weak Bear,Weak Bear,Strong Bear,Strong Bear,Strong Bear,2025-03-18
...,...,...,...,...,...,...,...,...,...,...
123,KUJI,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,Strong Bear,2025-03-18
124,PERP,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,2025-03-18
125,MKR,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Weak Bull,Weak Bull,Strong Bear,Weak Bull,2025-03-18
126,ALICE,Strong Bull,Strong Bear,Strong Bear,Weak Bear,Strong Bull,Strong Bear,Strong Bear,Weak Bear,2025-03-18


In [ ]:
# Filter for assets that are bullish in short term USD and BTC, and medium term
bullish_assets = summary_df[
    (summary_df['Short Term Trend (USD)'].isin(['Strong Bull', 'Weak Bull'])) &
    (summary_df['Short Term Trend (BTC)'].isin(['Strong Bull', 'Weak Bull'])) &
    (summary_df['Medium Term Trend (USD)'].isin(['Strong Bull', 'Weak Bull'])) |
    (summary_df['Medium Term Trend (BTC)'].isin(['Strong Bull', 'Weak Bull']))
].sort_values(by=['Short Term Trend (USD)', 'Medium Term Trend (USD)'], ascending=False)

bullish_assets

In [429]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Fetch classified data for asset
asset_id = 'maker'
classified_data = analyzer.asset_data.get(asset_id).get('classified_data')[
    ['date', 'Overall (USD)', 'Overall (BTC)', 
     'Short Term (USD)', 'Short Term (BTC)', 
     'Medium Term (USD)', 'Medium Term (BTC)', 
     'Long Term (USD)', 'Long Term (BTC)']
].copy()

classified_data.set_index('date', inplace=True)

# Simplified trend encoding (three states)
trend_encoding = {
    'Strong Bear': -1,
    'Weak Bear': -1,
    'Neutral': 0,
    'Weak Bull': 1,
    'Strong Bull': 1
}

classified_numeric = classified_data.replace(trend_encoding)
classified_numeric = classified_numeric[classified_numeric.index > '2022-01-01']

In [424]:
def calculate_transition_matrix(series):
    states = {-1: 'Bear', 0: 'Neutral', 1: 'Bull'}
    transition_counts = pd.crosstab(series.shift(1), series, normalize='index').fillna(0)
    transition_counts.index = transition_counts.index.map(states)
    transition_counts.columns = transition_counts.columns.map(states)
    return transition_counts

print(f"---- SHORT TERM ANALYSIS: {asset_id.upper()} ----", "\n")
transition_matrix_short_term_btc = calculate_transition_matrix(classified_numeric['Short Term (BTC)'])
print(transition_matrix_short_term_btc, "\n")
transition_matrix_short_term_usd = calculate_transition_matrix(classified_numeric['Short Term (USD)'])
print(transition_matrix_short_term_usd, "\n")

print("---- MEDIUM TERM ANALYSIS ----", "\n")
transition_matrix_medium_term_btc = calculate_transition_matrix(classified_numeric['Medium Term (BTC)'])
print(transition_matrix_medium_term_btc, "\n")
transition_matrix_medium_term_usd = calculate_transition_matrix(classified_numeric['Medium Term (USD)'])
print(transition_matrix_medium_term_usd, "\n")

print("---- LONG TERM ANALYSIS ----", "\n")
transition_matrix_long_term_btc = calculate_transition_matrix(classified_numeric['Long Term (BTC)'])
print(transition_matrix_long_term_btc, "\n")
transition_matrix_long_term_usd = calculate_transition_matrix(classified_numeric['Long Term (USD)'])
print(transition_matrix_long_term_usd, "\n")


---- SHORT TERM ANALYSIS: SOLANA ---- 

Short Term (BTC)      Bear      Bull
Short Term (BTC)                    
Bear              0.896660  0.103340
Bull              0.466981  0.533019 

Short Term (USD)      Bear      Bull
Short Term (USD)                    
Bear              0.911565  0.088435
Bull              0.270833  0.729167 

---- MEDIUM TERM ANALYSIS ---- 

Medium Term (BTC)      Bear      Bull
Medium Term (BTC)                    
Bear               0.946294  0.053706
Bull               0.209205  0.790795 

Medium Term (USD)      Bear      Bull
Medium Term (USD)                    
Bear               0.961806  0.038194
Bull               0.107843  0.892157 

---- LONG TERM ANALYSIS ---- 

Long Term (BTC)      Bear   Neutral      Bull
Long Term (BTC)                              
Bear             0.954751  0.030543  0.014706
Neutral          0.392857  0.452381  0.154762
Bull             0.039604  0.094059  0.866337 

Long Term (USD)      Bear   Neutral      Bull
Long Term 

In [ ]:
def calculate_trend_durability_stats(series):
    states = {-1: 'Bear', 0: 'Neutral', 1: 'Bull'}
    durations = {state: [] for state in states.values()}
    
    current_state = None
    duration = 0

    for val in series.dropna():
        label = states[val]
        if label == current_state:
            duration += 1
        else:
            if current_state is not None:
                durations[current_state].append(duration)
            current_state = label
            duration = 1
    
    # end
    if current_state:
        durations[current_state].append(duration)
    
    # Convert to median/mean/std
    output = {}
    for label, runs in durations.items():
        if len(runs) > 0:
            output[label] = {
                'Median': np.median(runs),
                'Mean': np.mean(runs),
                'Std': np.std(runs)
            }
        else:
            output[label] = {
                'Median': np.nan,
                'Mean': np.nan,
                'Std': np.nan
            }

    return pd.DataFrame(output).T

durability_stats = calculate_trend_durability_stats(classified_numeric['Overall (USD)'])
print(durability_stats)


In [ ]:
def plot_transition_matrix(matrix, title=''):
    G = nx.DiGraph()

    for from_state in matrix.index:
        for to_state in matrix.columns:
            weight = matrix.loc[from_state, to_state]
            if weight > 0:
                G.add_edge(from_state, to_state, weight=weight)

    plt.figure(figsize=(8,6))
    pos = nx.circular_layout(G)

    nx.draw(G, pos, with_labels=True, node_size=3000, node_color='lightgreen', arrowsize=20, font_size=14)

    # Edge labels (including self loops)
    edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}

    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red')

    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.show()

plot_transition_matrix(transition_matrix_long_term_btc, title=f'Transition Matrix with Persistence for {asset_id.upper()}')


In [ ]:
# Fetch raw asset price data
price_data = analyzer.asset_data[asset_id]['raw_data'][['date', 'close']].set_index('date').copy()

# Compute daily returns
price_data['Return'] = price_data['close'].pct_change()

# Merge trend state with returns
analysis_data = price_data.join(classified_numeric['Overall (USD)'].rename('Trend')).dropna()

# Plot histograms
def plot_return_histograms(data):
    states = {-1: 'Bear', 0: 'Neutral', 1: 'Bull'}
    plt.figure(figsize=(14,4))
    for state_val, state_name in states.items():
        plt.subplot(1,3,state_val+2)
        subset = data[data['Trend'] == state_val]['Return']
        plt.hist(subset, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        plt.title(f"{state_name} Market Returns")
        plt.xlabel("Daily Return")
        plt.ylabel("Frequency")
        plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_return_histograms(analysis_data)


In [ ]:
# Filter for assets with Weak Bull or Strong Bull in Short Term Trend (USD)
# AND Weak Bull or Strong Bull in Overall Trend (BTC)
bullish_assets = summary_df[
    (summary_df['Short Term Trend (USD)'].isin(['Weak Bull', 'Strong Bull'])) &
    (summary_df['Overall Trend (BTC)'].isin(['Weak Bull','Strong Bull']))
]
bullish_assets

In [430]:
alt_trend = analyzer.asset_data.get(asset_id).get('classified_data')[['date', 'open', 'high', 'low', 'close', 'Overall (BTC)', 'Overall (USD)', 'Short Term (BTC)', 'Short Term (USD)', 'Medium Term (BTC)', 'Medium Term (USD)', 'Long Term (BTC)', 'Long Term (USD)']]
alt_trend = alt_trend.rename(columns={'date': 'Date'})
alt_trend


,Date,open,high,low,close,Overall (BTC),Overall (USD),Short Term (BTC),Short Term (USD),Medium Term (BTC),Medium Term (USD),Long Term (BTC),Long Term (USD)
0,2017-12-21,1089.37,1089.37,1089.37,1089.37,Strong Bear,Strong Bear,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral
1,2017-12-22,1514.26,1514.26,1514.26,1514.26,Strong Bear,Strong Bear,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral
2,2017-12-23,873.83,873.83,873.83,873.83,Strong Bear,Strong Bear,Neutral,Neutral,Neutral,Neutral,Neutral,Neutral
3,2017-12-24,1042.82,1042.82,1042.82,1042.82,Weak Bear,Weak Bear,Strong Bull,Strong Bull,Neutral,Neutral,Neutral,Neutral
4,2017-12-25,1240.93,1240.93,1240.93,1240.93,Weak Bear,Weak Bear,Strong Bull,Strong Bull,Neutral,Neutral,Neutral,Neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2639,2025-03-13,1135.68,1143.07,1049.52,1131.91,Weak Bear,Strong Bear,Weak Bear,Weak Bear,Weak Bull,Strong Bear,Strong Bear,Strong Bear
2641,2025-03-15,1133.37,1209.50,1130.51,1171.21,Weak Bull,Weak Bear,Weak Bull,Weak Bull,Weak Bull,Strong Bear,Strong Bear,Strong Bear
2642,2025-03-16,1169.97,1224.85,1168.51,1217.31,Weak Bull,Weak Bear,Weak Bull,Weak Bull,Weak Bull,Strong Bear,Strong Bear,Strong Bear
2643,2025-03-17,1214.62,1232.97,1159.59,1178.35,Weak Bull,Weak Bear,Weak Bull,Weak Bull,Weak Bull,Strong Bear,Strong Bear,Strong Bear


In [ ]:
#btc_strategy = pd.read_csv('hmm_btc_strategy_14_03_2025.csv')[['Date','state', 'Close']]

from scripts.hmm_btc import RegimeSwitchModel

model = RegimeSwitchModel('bitcoin', strategy='long-only', train_pct=0.8, confidence_threshold=0)

version_to_load = "20250317" 

recommendations_df = model.load_model_and_predict(version=version_to_load)

recommendations_df

In [ ]:
df = model.data[['Close', 'Open']]
btc_data = pd.merge(recommendations_df, df, on='Date', how='inner')
btc_data.rename(columns={'Close': 'btc_close', 'Open': 'btc_open', 'Decision': 'btc_state'}, inplace=True)

In [ ]:
results = model.simulate_trading(recommendations_df, save_path=None)
model.evaluate(results)

In [431]:

data = pd.merge(alt_trend, btc_data, on='Date', how='inner')
data

,Date,open,high,low,close,Overall (BTC),Overall (USD),Short Term (BTC),Short Term (USD),Medium Term (BTC),Medium Term (USD),Long Term (BTC),Long Term (USD),btc_state,Confidence,btc_close,btc_open
0,2023-12-17,1311.12,1334.58,1303.24,1327.57,Weak Bear,Weak Bear,Weak Bull,Weak Bear,Strong Bear,Strong Bear,Strong Bear,Weak Bull,Flat,1.000000,42247.0,41890.0
1,2023-12-18,1328.45,1346.64,1315.63,1316.03,Strong Bear,Weak Bear,Weak Bear,Weak Bear,Strong Bear,Strong Bear,Strong Bear,Weak Bull,Flat,0.994493,41411.0,42248.0
2,2023-12-19,1316.39,1318.08,1255.48,1287.62,Strong Bear,Strong Bear,Weak Bear,Weak Bear,Strong Bear,Strong Bear,Strong Bear,Neutral,Flat,0.999937,42684.0,41355.0
3,2023-12-20,1288.16,1307.56,1262.19,1289.37,Strong Bear,Strong Bear,Weak Bear,Weak Bear,Strong Bear,Strong Bear,Strong Bear,Neutral,Flat,0.999862,42250.0,42609.0
4,2023-12-21,1289.80,1316.94,1274.94,1285.63,Strong Bear,Strong Bear,Weak Bear,Weak Bear,Strong Bear,Strong Bear,Strong Bear,Neutral,Flat,0.999877,43634.0,42276.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
452,2025-03-13,1135.68,1143.07,1049.52,1131.91,Weak Bear,Strong Bear,Weak Bear,Weak Bear,Weak Bull,Strong Bear,Strong Bear,Strong Bear,Flat,1.000000,83884.0,82943.0
453,2025-03-15,1133.37,1209.50,1130.51,1171.21,Weak Bull,Weak Bear,Weak Bull,Weak Bull,Weak Bull,Strong Bear,Strong Bear,Strong Bear,Flat,0.999974,83972.0,81015.0
454,2025-03-16,1169.97,1224.85,1168.51,1217.31,Weak Bull,Weak Bear,Weak Bull,Weak Bull,Weak Bull,Strong Bear,Strong Bear,Strong Bear,Flat,0.999998,84392.0,83955.0
455,2025-03-17,1214.62,1232.97,1159.59,1178.35,Weak Bull,Weak Bear,Weak Bull,Weak Bull,Weak Bull,Strong Bear,Strong Bear,Strong Bear,Flat,0.999971,82611.0,84345.0


In [432]:
# Define the strategy combining alt trend signals and BTC HMM model
def combined_strategy_backtest(data, initial_investment=1000, risk_free_rate=   0.0):
    """
    Hybrid trading strategy:
    - By default, follow the BTC HMM strategy (go long BTC when HMM says "Long")
    - When alt-BTC combined conditions are also met, split investment 50-50 between alt and BTC
    - Stay in cash when BTC HMM model doesn't signal "Long"
    
    Args:
        data: DataFrame with trend classifications, price data, and BTC state
        initial_investment: Initial portfolio value (default: $1000)
        risk_free_rate: Annual risk-free rate (default: 0.0)
    
    Returns:
        DataFrame with strategy signals and performance metrics
    """
    # Create a copy of the data to avoid modifying the original
    strategy_data = data.copy()
    
    # Create a BTC signal column (1 when HMM says Long, 0 otherwise)
    strategy_data['btc_signal'] = 0
    strategy_data.loc[strategy_data['btc_state'] == 'Long', 'btc_signal'] = 1
    
    # Generate signals based on our combined conditions for alt strategy
    strategy_conditions = (
        ((strategy_data['Short Term (USD)'].isin(['Weak Bull', 'Strong Bull'])) & 
        (strategy_data['Short Term (BTC)'].isin(['Weak Bull', 'Strong Bull'])) ) &
        (strategy_data['Medium Term (BTC)'].isin(['Weak Bull', 'Strong Bull'])) &
        (strategy_data['Long Term (USD)'].isin(['Weak Bull', 'Strong Bull'])) & 
        (strategy_data['btc_state'] == 'Long')
    )
    
    strategy_data['alt_signal'] = 0
    strategy_data.loc[strategy_conditions, 'alt_signal'] = 1
    
    # Calculate daily returns
    strategy_data['daily_return'] = strategy_data['close'].pct_change()
    strategy_data['btc_daily_return'] = strategy_data['btc_close'].pct_change()
    
    # Calculate strategy returns with a 1-day lag to avoid look-ahead bias
    # BTC strategy returns
    strategy_data['btc_strategy_return'] = strategy_data['btc_signal'].shift(1) * strategy_data['btc_daily_return']
    
    # Alt-only strategy returns
    strategy_data['alt_strategy_return'] = strategy_data['alt_signal'].shift(1) * strategy_data['daily_return']
    
    # Calculate hybrid strategy returns:
    # - When alt_signal is 1, allocate 50% to alt and 50% to BTC
    # - When alt_signal is 0 but btc_signal is 1, allocate 100% to BTC
    # - When both signals are 0, stay in cash (0% return)
    strategy_data['hybrid_return'] = 0.0
    
    # Case 1: When alt_signal is active, invest 50-50
    alt_signal_active = strategy_data['alt_signal'].shift(1) == 1
    strategy_data.loc[alt_signal_active, 'hybrid_return'] = (
        0.5 * strategy_data.loc[alt_signal_active, 'daily_return'] + 
        0.5 * strategy_data.loc[alt_signal_active, 'btc_daily_return']
    )
    
    # Case 2: When only BTC signal is active, invest 100% in BTC
    btc_only_active = (strategy_data['alt_signal'].shift(1) == 0) & (strategy_data['btc_signal'].shift(1) == 1)
    strategy_data.loc[btc_only_active, 'hybrid_return'] = strategy_data.loc[btc_only_active, 'btc_daily_return']
    
    # Calculate portfolio values
    strategy_data['buy_hold_value'] = initial_investment * (1 + strategy_data['daily_return']).cumprod()
    strategy_data['btc_value'] = initial_investment * (1 + strategy_data['btc_daily_return']).cumprod()
    strategy_data['btc_strategy_value'] = initial_investment * (1 + strategy_data['btc_strategy_return']).cumprod()
    strategy_data['alt_strategy_value'] = initial_investment * (1 + strategy_data['alt_strategy_return']).cumprod()
    strategy_data['hybrid_strategy_value'] = initial_investment * (1 + strategy_data['hybrid_return']).cumprod()
    
    # Correctly calculate drawdowns
    strategy_data['buy_hold_peak'] = strategy_data['buy_hold_value'].cummax()
    strategy_data['btc_peak'] = strategy_data['btc_value'].cummax()
    strategy_data['btc_strategy_peak'] = strategy_data['btc_strategy_value'].cummax()
    strategy_data['alt_strategy_peak'] = strategy_data['alt_strategy_value'].cummax()
    strategy_data['hybrid_strategy_peak'] = strategy_data['hybrid_strategy_value'].cummax()
    
    strategy_data['buy_hold_drawdown'] = (strategy_data['buy_hold_value'] - strategy_data['buy_hold_peak']) / strategy_data['buy_hold_peak']
    strategy_data['btc_drawdown'] = (strategy_data['btc_value'] - strategy_data['btc_peak']) / strategy_data['btc_peak']
    strategy_data['btc_strategy_drawdown'] = (strategy_data['btc_strategy_value'] - strategy_data['btc_strategy_peak']) / strategy_data['btc_strategy_peak']
    strategy_data['alt_strategy_drawdown'] = (strategy_data['alt_strategy_value'] - strategy_data['alt_strategy_peak']) / strategy_data['alt_strategy_peak']
    strategy_data['hybrid_strategy_drawdown'] = (strategy_data['hybrid_strategy_value'] - strategy_data['hybrid_strategy_peak']) / strategy_data['hybrid_strategy_peak']
    
    return strategy_data

# Run the backtest with initial investment of $1000
initial_investment = 1000
risk_free_rate = 0.0  # Assuming 0% risk-free rate for crypto
backtest_results = combined_strategy_backtest(data, initial_investment, risk_free_rate)

# Calculate Sharpe and Sortino ratios
def calculate_risk_metrics(returns, risk_free_rate=0.0):
    """Calculate Sharpe and Sortino ratios for a series of returns"""
    # Remove NaN values
    returns = returns.dropna()
    
    if len(returns) == 0 or returns.std() == 0:
        return 0, 0
    
    # Calculate excess returns (assuming daily risk-free rate)
    daily_rf_rate = (1 + risk_free_rate) ** (1/365) - 1
    excess_returns = returns - daily_rf_rate
    
    # Calculate Sharpe Ratio (annualized)
    sharpe_ratio = np.sqrt(365) * excess_returns.mean() / excess_returns.std()
    
    # Calculate Sortino Ratio (annualized)
    downside_returns = excess_returns[excess_returns < 0]
    sortino_ratio = np.sqrt(365) * excess_returns.mean() / downside_returns.std() if len(downside_returns) > 0 and downside_returns.std() > 0 else np.inf
    
    return sharpe_ratio, sortino_ratio

# Calculate risk metrics for all approaches
buy_hold_sharpe, buy_hold_sortino = calculate_risk_metrics(backtest_results['daily_return'].dropna(), risk_free_rate)
btc_sharpe, btc_sortino = calculate_risk_metrics(backtest_results['btc_daily_return'].dropna(), risk_free_rate)
btc_strategy_sharpe, btc_strategy_sortino = calculate_risk_metrics(backtest_results['btc_strategy_return'].dropna(), risk_free_rate)
alt_strategy_sharpe, alt_strategy_sortino = calculate_risk_metrics(backtest_results['alt_strategy_return'].dropna(), risk_free_rate)
hybrid_sharpe, hybrid_sortino = calculate_risk_metrics(backtest_results['hybrid_return'].dropna(), risk_free_rate)

# Visualize the strategy performance with portfolio values
import plotly.express as px

fig = px.line(backtest_results, x='Date', 
             y=['buy_hold_value', 'btc_value', 'btc_strategy_value', 'alt_strategy_value', 'hybrid_strategy_value'],
             title='Hybrid Strategy Backtest Performance',
             labels={'value': 'Portfolio Value ($)', 'variable': 'Strategy'},
             color_discrete_map={
                 'buy_hold_value': 'gray',
                 'btc_value': 'orange',
                 'btc_strategy_value': 'blue',
                 'alt_strategy_value': 'red',
                 'hybrid_strategy_value': 'green'
             })

fig.update_layout(
    xaxis_title='',
    yaxis_title='Portfolio Value ($)',
    template='plotly_white',
    height=600,
    width=1000,
    legend=dict(
        title=None,
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

# Update legend labels
fig.for_each_trace(lambda t: t.update(
    name={
        'buy_hold_value': 'Alt Buy & Hold',
        'btc_value': 'BTC Buy & Hold',
        'btc_strategy_value': 'BTC HMM Strategy',
        'alt_strategy_value': 'Alt-only Strategy',
        'hybrid_strategy_value': 'Hybrid Strategy (BTC/50-50)'
    }[t.name]
))

fig.show()

# Calculate percentage of time in market
total_days = len(backtest_results)
alt_invested_days = backtest_results['alt_signal'].sum()
btc_invested_days = backtest_results['btc_signal'].sum()
hybrid_both_days = ((backtest_results['alt_signal'] == 1) & (backtest_results['btc_signal'] == 1)).sum()
hybrid_btc_only_days = ((backtest_results['alt_signal'] == 0) & (backtest_results['btc_signal'] == 1)).sum()
hybrid_total_days = hybrid_both_days + hybrid_btc_only_days

alt_percent_invested = alt_invested_days / total_days * 100
btc_percent_invested = btc_invested_days / total_days * 100
hybrid_percent_invested = hybrid_total_days / total_days * 100
percent_5050_allocation = hybrid_both_days / hybrid_total_days * 100 if hybrid_total_days > 0 else 0

# Calculate total and annualized returns
days_in_market = (backtest_results['Date'].iloc[-1] - backtest_results['Date'].iloc[0]).days
annual_factor = 365 / days_in_market

final_buy_hold_value = backtest_results['buy_hold_value'].iloc[-1]
final_btc_value = backtest_results['btc_value'].iloc[-1]
final_btc_strategy_value = backtest_results['btc_strategy_value'].iloc[-1]
final_alt_strategy_value = backtest_results['alt_strategy_value'].iloc[-1]
final_hybrid_value = backtest_results['hybrid_strategy_value'].iloc[-1]

buy_hold_return = (final_buy_hold_value / initial_investment) - 1
btc_return = (final_btc_value / initial_investment) - 1
btc_strategy_return = (final_btc_strategy_value / initial_investment) - 1
alt_strategy_return = (final_alt_strategy_value / initial_investment) - 1
hybrid_return = (final_hybrid_value / initial_investment) - 1

annualized_buy_hold = (1 + buy_hold_return) ** annual_factor - 1
annualized_btc = (1 + btc_return) ** annual_factor - 1
annualized_btc_strategy = (1 + btc_strategy_return) ** annual_factor - 1
annualized_alt_strategy = (1 + alt_strategy_return) ** annual_factor - 1
annualized_hybrid = (1 + hybrid_return) ** annual_factor - 1

# Get max drawdowns
max_bh_drawdown = backtest_results['buy_hold_drawdown'].min()
max_btc_drawdown = backtest_results['btc_drawdown'].min()
max_btc_strategy_drawdown = backtest_results['btc_strategy_drawdown'].min()
max_alt_strategy_drawdown = backtest_results['alt_strategy_drawdown'].min()
max_hybrid_drawdown = backtest_results['hybrid_strategy_drawdown'].min()

# Calculate return/drawdown ratios
return_dd_bh = annualized_buy_hold / abs(max_bh_drawdown) if max_bh_drawdown != 0 else float('inf')
return_dd_btc = annualized_btc / abs(max_btc_drawdown) if max_btc_drawdown != 0 else float('inf')
return_dd_btc_strategy = annualized_btc_strategy / abs(max_btc_strategy_drawdown) if max_btc_strategy_drawdown != 0 else float('inf')
return_dd_alt_strategy = annualized_alt_strategy / abs(max_alt_strategy_drawdown) if max_alt_strategy_drawdown != 0 else float('inf')
return_dd_hybrid = annualized_hybrid / abs(max_hybrid_drawdown) if max_hybrid_drawdown != 0 else float('inf')

# Print performance summary
print(f"Hybrid Strategy Performance Summary:")
print(f"Period: {backtest_results['Date'].iloc[0].date()} to {backtest_results['Date'].iloc[-1].date()} ({days_in_market} days)")
print(f"Initial Investment: ${initial_investment:.2f}")
print(f"\nStrategy Allocation Details:")
print(f"BTC HMM Strategy - Time invested in market: {btc_percent_invested:.2f}%")
print(f"Alt-only Strategy - Time invested in market: {alt_percent_invested:.2f}%")
print(f"Hybrid Strategy - Time invested in market: {hybrid_percent_invested:.2f}%")
print(f"  - Of which, 50-50 allocation: {percent_5050_allocation:.2f}%")
print(f"  - Of which, 100% BTC allocation: {100-percent_5050_allocation:.2f}%")

print("\nTotal Returns:")
print(f"Alt Buy & Hold: ${final_buy_hold_value:.2f} (Total return: {buy_hold_return:.2%})")
print(f"BTC Buy & Hold: ${final_btc_value:.2f} (Total return: {btc_return:.2%})")
print(f"BTC HMM Strategy: ${final_btc_strategy_value:.2f} (Total return: {btc_strategy_return:.2%})")
print(f"Alt-only Strategy: ${final_alt_strategy_value:.2f} (Total return: {alt_strategy_return:.2%})")
print(f"Hybrid Strategy: ${final_hybrid_value:.2f} (Total return: {hybrid_return:.2%})")

print("\nAnnualized Returns:")
print(f"Alt Buy & Hold: {annualized_buy_hold:.2%}")
print(f"BTC Buy & Hold: {annualized_btc:.2%}")
print(f"BTC HMM Strategy: {annualized_btc_strategy:.2%}")
print(f"Alt-only Strategy: {annualized_alt_strategy:.2%}")
print(f"Hybrid Strategy: {annualized_hybrid:.2%}")

print("\nRisk Metrics:")
print(f"Max Drawdown - Alt Buy & Hold: {max_bh_drawdown:.2%}")
print(f"Max Drawdown - BTC Buy & Hold: {max_btc_drawdown:.2%}")
print(f"Max Drawdown - BTC HMM Strategy: {max_btc_strategy_drawdown:.2%}")
print(f"Max Drawdown - Alt-only Strategy: {max_alt_strategy_drawdown:.2%}")
print(f"Max Drawdown - Hybrid Strategy: {max_hybrid_drawdown:.2%}")

print("\nSharpe Ratio:")
print(f"Alt Buy & Hold: {buy_hold_sharpe:.2f}")
print(f"BTC Buy & Hold: {btc_sharpe:.2f}")
print(f"BTC HMM Strategy: {btc_strategy_sharpe:.2f}")
print(f"Alt-only Strategy: {alt_strategy_sharpe:.2f}")
print(f"Hybrid Strategy: {hybrid_sharpe:.2f}")

print("\nSortino Ratio:")
print(f"Alt Buy & Hold: {buy_hold_sortino:.2f}")
print(f"BTC Buy & Hold: {btc_sortino:.2f}")
print(f"BTC HMM Strategy: {btc_strategy_sortino:.2f}")
print(f"Alt-only Strategy: {alt_strategy_sortino:.2f}")
print(f"Hybrid Strategy: {hybrid_sortino:.2f}")

print("\nReturn/MaxDD Ratio:")
print(f"Alt Buy & Hold: {return_dd_bh:.2f}")
print(f"BTC Buy & Hold: {return_dd_btc:.2f}")
print(f"BTC HMM Strategy: {return_dd_btc_strategy:.2f}")
print(f"Alt-only Strategy: {return_dd_alt_strategy:.2f}")
print(f"Hybrid Strategy: {return_dd_hybrid:.2f}")

# Plot drawdowns to visualize risk reduction
fig_drawdown = px.line(backtest_results, x='Date', 
                      y=['buy_hold_drawdown', 'btc_drawdown', 'btc_strategy_drawdown', 'alt_strategy_drawdown', 'hybrid_strategy_drawdown'],
                      title='Drawdown Comparison',
                      labels={'value': 'Drawdown', 'variable': 'Strategy'},
                      color_discrete_map={
                          'buy_hold_drawdown': 'gray',
                          'btc_drawdown': 'orange',
                          'btc_strategy_drawdown': 'blue',
                          'alt_strategy_drawdown': 'red',
                          'hybrid_strategy_drawdown': 'green'
                      })

fig_drawdown.update_layout(
    xaxis_title='',
    yaxis_title='Drawdown',
    template='plotly_white',
    height=400,
    width=1000
)

# Update legend labels
fig_drawdown.for_each_trace(lambda t: t.update(
    name={
        'buy_hold_drawdown': 'Alt Buy & Hold',
        'btc_drawdown': 'BTC Buy & Hold',
        'btc_strategy_drawdown': 'BTC HMM Strategy',
        'alt_strategy_drawdown': 'Alt-only Strategy',
        'hybrid_strategy_drawdown': 'Hybrid Strategy (BTC/50-50)'
    }[t.name]
))

fig_drawdown.show()

Hybrid Strategy Performance Summary:
Period: 2023-12-17 to 2025-03-18 (457 days)
Initial Investment: $1000.00

Strategy Allocation Details:
BTC HMM Strategy - Time invested in market: 61.05%
Alt-only Strategy - Time invested in market: 10.07%
Hybrid Strategy - Time invested in market: 61.05%
  - Of which, 50-50 allocation: 16.49%
  - Of which, 100% BTC allocation: 83.51%

Total Returns:
Alt Buy & Hold: $910.83 (Total return: -8.92%)
BTC Buy & Hold: $1990.08 (Total return: 99.01%)
BTC HMM Strategy: $3187.89 (Total return: 218.79%)
Alt-only Strategy: $999.15 (Total return: -0.09%)
Hybrid Strategy: $3460.74 (Total return: 246.07%)

Annualized Returns:
Alt Buy & Hold: -7.19%
BTC Buy & Hold: 73.26%
BTC HMM Strategy: 152.43%
Alt-only Strategy: -0.07%
Hybrid Strategy: 169.54%

Risk Metrics:
Max Drawdown - Alt Buy & Hold: -77.31%
Max Drawdown - BTC Buy & Hold: -26.23%
Max Drawdown - BTC HMM Strategy: -15.00%
Max Drawdown - Alt-only Strategy: -24.11%
Max Drawdown - Hybrid Strategy: -13.69%

Sha

In [ ]:
import plotly.express as px

# Define the trend order and color mapping
trend_order = ['Strong Bull', 'Weak Bull', 'Neutral', 'Weak Bear', 'Strong Bear']
color_map = {
    'Strong Bull': 'darkgreen',
    'Weak Bull': 'lightgreen',
    'Neutral': 'gray',
    'Weak Bear': 'orange',
    'Strong Bear': 'red'
}

# Create a scatter plot of Overall trend classification through time using plotly
fig = px.scatter(analyzer.classified_data, x='date', y='Overall', 
                 title='Overall Trend Classification Through Time',
                 color='Overall',
                 color_discrete_map=color_map,
                 category_orders={'Overall': trend_order})

fig.update_layout(
    xaxis_title='',
    yaxis_title='Trend Classification',
    template='plotly_white',
    height=600,
    width=1000,
    yaxis=dict(
        categoryorder='array',
        categoryarray=trend_order
    )
)

fig.update_traces(marker=dict(size=4))
fig.show()

In [ ]:
# Define a simple trading strategy based on trend classification
def trend_based_strategy(data):
    """
    Trading strategy that:
    - Goes long at market open if previous day's close was classified as 'weak bull' or 'strong bull'
    - Stays in cash otherwise
    
    Args:
        data: DataFrame with trend classifications and price data
    
    Returns:
        DataFrame with strategy signals and performance metrics
    """
    # Create a copy of the data to avoid modifying the original
    strategy_data = data.copy()
    
    # Create a signal column (1 for long, 0 for cash)
    strategy_data['signal'] = 0
    
    # Generate signals based on previous day's classification
    bullish_conditions = (strategy_data['Short Term'].shift(1).isin(['Weak Bull', 'Strong Bull']))

    #bearish_conditions = (strategy_data['Overall'].shift(1).isin(['Strong Bear']))
    strategy_data.loc[bullish_conditions, 'signal'] = 1

    #strategy_data.loc[bearish_conditions, 'signal'] = -1    
    # Calculate returns
    strategy_data['daily_return'] = strategy_data['close'].pct_change()
    # Calculate strategy returns (signal from previous day * today's return)
    strategy_data['strategy_return'] = strategy_data['signal'] * strategy_data['daily_return']
    
    # Calculate cumulative returns
    strategy_data['cumulative_return'] = (1 + strategy_data['daily_return']).cumprod() - 1
    strategy_data['strategy_cumulative_return'] = (1 + strategy_data['strategy_return']).cumprod() - 1
    
    return strategy_data

# Apply the strategy to our classified data
strategy_results = trend_based_strategy(analyzer.classified_data)

# Visualize the strategy performance
fig = px.line(strategy_results, x='date', y=['cumulative_return', 'strategy_cumulative_return'],
              title='Trend-Based Trading Strategy Performance',
              labels={'value': 'Cumulative Return', 'variable': 'Strategy'},
              color_discrete_map={
                  'cumulative_return': 'gray',
                  'strategy_cumulative_return': 'blue'
              })

fig.update_layout(
    xaxis_title='',
    yaxis_title='Cumulative Return',
    template='plotly_white',
    height=600,
    width=1000,
    legend=dict(
        title=None,
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

# Update legend labels
fig.for_each_trace(lambda t: t.update(name='Buy & Hold' if t.name == 'cumulative_return' else 'Trend Strategy'))

fig.show()

# Calculate performance metrics
total_days = len(strategy_results)
invested_days = strategy_results['signal'].sum()
percent_invested = invested_days / total_days * 100

# Calculate annualized returns
days_in_market = (strategy_results['date'].iloc[-1] - strategy_results['date'].iloc[0]).days
annual_factor = 365 / days_in_market
buy_hold_return = strategy_results['cumulative_return'].iloc[-1]
strategy_return = strategy_results['strategy_cumulative_return'].iloc[-1]
annualized_buy_hold = (1 + buy_hold_return) ** annual_factor - 1
annualized_strategy = (1 + strategy_return) ** annual_factor - 1

# Print performance summary
print(f"Strategy Performance Summary:")
print(f"Period: {strategy_results['date'].iloc[0].date()} to {strategy_results['date'].iloc[-1].date()} ({days_in_market} days)")
print(f"Time invested in market: {percent_invested:.2f}%")
print(f"Buy & Hold return: {buy_hold_return:.2%} (Annualized: {annualized_buy_hold:.2%})")
print(f"Strategy return: {strategy_return:.2%} (Annualized: {annualized_strategy:.2%})")
print(f"Outperformance: {strategy_return - buy_hold_return:.2%}")


In [ ]:
df = pd.read_csv('/Users/valter.rebelo/MissionControl/data/micro/candleData/akash-network_candles.csv')

In [ ]:
import ta 

df['sma_3'] = ta.trend.SMAIndicator(close=df['close'], window=3).sma_indicator()
df['sma_7'] = ta.trend.SMAIndicator(close=df['close'], window=7).sma_indicator()
df['sma_30'] = ta.trend.SMAIndicator(close=df['close'], window=30).sma_indicator()
df['sma_365'] = ta.trend.SMAIndicator(close=df['close'], window=365).sma_indicator()

# Price to MA ratios
df['price_sma3_ratio'] = df['close'] / df['sma_3']
df['price_sma7_ratio'] = df['close'] / df['sma_7']
df['price_sma30_ratio'] = df['close'] / df['sma_30']
df['price_sma365_ratio'] = df['close'] / df['sma_365']

df['sma_3_low'] = df['low'].rolling(window=3).min()
df['7day_low'] = df['low'].rolling(window=7).min()
df['14day_low'] = df['low'].rolling(window=14).min()
df['30day_low'] = df['low'].rolling(window=30).min()
    
df['sma_3_high'] = df['high'].rolling(window=3).max()
df['7day_high'] = df['high'].rolling(window=7).max()
df['14day_high'] = df['high'].rolling(window=14).max()
df['30day_high'] = df['high'].rolling(window=30).max()


df['support'] = (df['7day_low'] + df['14day_low'] + df['30day_low'] + df['sma_3_low']) / 4
df['resistance'] = (df['7day_high'] + df['14day_high'] + df['30day_high'] + df['sma_3_high']) / 4

df['support_relative'] = df['close'] / df['support']
df['resistance_relative'] = df['close'] / df['resistance'] 

In [ ]:
df[['resistance_relative','support_relative']].hist(bins=50)


In [ ]:
df[['price_sma30_ratio', 'price_sma365_ratio', 'price_sma3_ratio', 'price_sma7_ratio']].hist(bins=50)